In [ ]:
import os

import json
import pandas as pd
import random
import time

from datetime import datetime
from dotenv import load_dotenv

from pathlib import Path
import sys

In [ ]:
MODULES_PATH = Path("../modules").resolve()

if str(MODULES_PATH) not in sys.path:
    sys.path.insert(0, str(MODULES_PATH))

import corpus
import api

from dev import rel, print_epi_summary
from data_config import DATA_CONFIG, FEW_SHOT_PATH
from few_shot import get_few_shot_examples

import pipeline

# Config.

### Development Config.

In [ ]:
load_dotenv()

# ---------- Set seed to 42 ----------
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)

# ---------- Print confirmation ----------
print(
    f"Set random seed to {SEED} at "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M')}"
)

Set random seed to 42 at2026-08-08 21:40


### EPI Config.

In [18]:
# ---------- Manual EPI config. entry ----------
EPI_NUM = "006"
DATASET_SPLIT = "train"

# Corpus Loading

### Load Corpus

In [ ]:
# ---------- Get abstracts path ----------
ABSTRACTS_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["abstracts"]
SAMPLE_SIZE = DATA_CONFIG[DATASET_SPLIT]["sample_size"]

# ---------- Load corpus from path ----------
abstracts_corpus = corpus.load_corpus(ABSTRACTS_PATH, sample_size=SAMPLE_SIZE)
print(f"Abstracts dataset length: {len(abstracts_corpus)}")

Abstracts dataset length: 35


### Get Ground Truths
* Optional sample functionality

In [20]:
# ---------- Manually Set Ground Truth Path ----------
GT_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["ground_truths"]

# ---------- Fetch BioRED Ground Truths ----------
abstracts_ground_truths = corpus.get_corpus_gt_csv(abstracts_corpus, GT_PATH)

In [21]:
# ---------- Turn ground truth DataFrame into structured dict ----------
ground_truths = corpus.get_gt_dict(abstracts_ground_truths)

### Few-Shot Construction

In [22]:
# ---------- If FS block does not exist, create FS block, else pass ----------
if not os.path.exists(FEW_SHOT_PATH):

    few_shot_block = get_few_shot_examples(
        path_to_train_set=DATA_CONFIG["train"]["paths"]["abstracts"],
        path_to_train_gts=DATA_CONFIG["train"]["paths"]["ground_truths"],
        biored_train_samples=abstracts_corpus,
        few_shot_export_path=FEW_SHOT_PATH
    )
    print(f"Generated and exported new few-shot block to '{FEW_SHOT_PATH}'")
else:
    with open(FEW_SHOT_PATH) as f:
        few_shot_block = f.read()
    print(f"Imported existing few-shot block from '{FEW_SHOT_PATH}' at {datetime.now().strftime('%Y-%m-%d %H:%M')}.")

Imported existing few-shot block from '../../data/few_shot/few_shot_block.txt' at 2026-08-08 21:43.


### Import BioRED Extraction Guidelines

In [23]:
# ---------- Import BioRED guidelines text file for prompt refinement ----------
with open("../../data/processed/biored/guidelines.txt", "r", encoding="utf-8") as f:
    biored_ext_guidelines = f.read()

# ---------- Print preview ----------
print(f"{biored_ext_guidelines[:500]}...")

## Guideline of the entities

### General rules
- Annotate all the spans of all the six concept types.
- The full text can be accessed to clarify the concept spans and identifiers.
- The abbreviation and its long form should be annotated separately if possible. prostaglandin E2 (PGE2) in the text, “prostaglandin E2” and “PGE2” should be both annotated to chemicals with the same identifier (D015232).
- Annotate both the full name and abbreviation in one entity, if the boundary of the entity cover...


# OpenAI Luna API Call

### EPI Setup

In [24]:
# ---------- Create EPI setup dictionary ----------
# - Keys: "dataset", "id", "eval_version", "notes", "reuse_api_call", "prompt"
epi_setup = pipeline.setup_epi(EPI_NUM, DATASET_SPLIT)

# ---------- Print summary ----------
print_epi_summary(
    epi_num=EPI_NUM,
    prompt_version=epi_setup["prompt"]["version"],
    eval_version=epi_setup["eval_version"],
    notes=epi_setup["notes"],
    reuse_api_call=epi_setup["reuse_api_call"]
)

Cell ran at 2026-08-08 21:43 for epi_006
 - Prompt version: v3
 - Evaluation version: v4
 - Notes: Relationship matching now order-invariant across strict/relaxed/cosine.
 - Reuse API Call: True


### API Call

In [25]:
epi_setup.keys()

dict_keys(['dataset', 'id', 'eval_version', 'notes', 'reuse_api_call', 'prompt'])

In [26]:
# ---------- Create client ----------
client = api.create_client()

# ---------- If prompt version is different from previous EPI, call API, else pass ----------
if not epi_setup["reuse_api_call"]:

    print(f"Running API call for {epi_setup["id"]}...")

    # Fetch raw prompt template
    prompt_template = epi_setup["prompt"]["template"]

    # Begin timer
    start_time = time.perf_counter()

    # Initiate outputs list
    outputs = []

    # Begin looping through abstract dataset rows - one call per row
    for index, row in abstracts_corpus.iterrows():
        abstract = row["abstract"]

        # Replace prompt template's placeholders with abstract, few_shot & guidelines
        prompt = (
            prompt_template
            .replace("{abstract}", abstract)
            .replace("{few_shot_block}", few_shot_block)
            .replace("{biored_ext_guidelines}", biored_ext_guidelines)
        )

        # Store row's response
        response = client.responses.create(
            model="gpt-5.6-luna",
            input=prompt
        )

        # Append response to outputs list
        outputs.append({
            "pmid": row["pmid"],
            "output": response.output_text
        })

    # End time, store elapsed time & print result
    elapsed_seconds = time.perf_counter() - start_time
    print(f"API calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts\n")

    print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f" - Prompt verion: {epi_setup["prompt"]["version"]}")
    print(f" - Evalaution verion: {epi_setup["eval_version"]}")
    print(f" - Run notes: {epi_setup["notes"]}")
    print(f" - Elapsed time: {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f}")
else:
    prev_epi_id = f"epi_{int(EPI_NUM) - 1:03d}"

    with open(f"{DATA_CONFIG[DATASET_SPLIT]["paths"]["epis"]}/{prev_epi_id}.json") as f:
        prev_epi_log = json.load(f)

    outputs = prev_epi_log["outputs"]
    prompt_template = prev_epi_log["prompt"]
    elapsed_seconds = prev_epi_log["time_taken"]

    print(f"Reused API call from {prev_epi_id}.")

output_info = {
    "epi_id": epi_setup["id"],
    "outputs": outputs,
    "time_taken": elapsed_seconds,
    "raw_prompt": prompt_template,
    "epi_notes": epi_setup["notes"],
    "prompt_version": epi_setup["prompt"]["version"],
    "eval_version": epi_setup["eval_version"],
    "export_path": DATA_CONFIG[DATASET_SPLIT]["paths"]["epis"]
}

Created client:
 - Timeout: 60
 - Max retries: 0

Reused API call from epi_005.


In [27]:
# ---------- Parse outputs, evaluate extractions, export EPI results ----------
epi_log = pipeline.process_epi(output_info, ground_truths, dataset=epi_setup["dataset"])

Parsed 35 extractions, 0 failed to parse as JSON
Saved epi_006 to ../../data/results/epis/biored_train


# Error Analysis

### Relations

In [28]:
# ---------- False positives ----------
relations_fp = epi_log["errors"]["errors"]["relations"]["false_positives"]

print(f"Count: {len(relations_fp)}")
display(relations_fp)

Count: 174


[[19521089, '5-httlpr', 'Association', 'slc6a4'],
 [29222418, 'heb', 'Positive_Correlation', 'il-17'],
 [17192049, 'a to g transition in exon7', 'Association', 'prostate cancer'],
 [16120104, 'per2', 'Association', 'asps'],
 [19108278, '(+)-propranolol', 'Comparison', '(-)-propranolol'],
 [28428256, 'lps', 'Positive_Correlation', 'icam1'],
 [19108278, '(-)-propranolol', 'Comparison', 'procaine'],
 [28428256, 'ep4', 'Positive_Correlation', 'rap1'],
 [28428256, 'pga2', 'Positive_Correlation', 'cortactin'],
 [15099351, 'n157k', 'Association', 'pcsk9'],
 [19918264, 'arggly genotype', 'Association', 'prostate cancer'],
 [15970799, '521t>c', 'Association', 'slco1b1*5'],
 [24309294, 'cck-8', 'Bind', 'cck2 receptor'],
 [16574712, 'mdma', 'Positive_Correlation', 'set shifting'],
 [18768591, 'enac', 'Positive_Correlation', 'volume retention'],
 [16288197, '144 g>a', 'Association', 'q48h'],
 [18808529, 'dystrophin', 'Bind', 'actin'],
 [16288197, 'congenital microcoria', 'Positive_Correlation', 'g

In [29]:
# ---------- False negatives ----------
relations_fn = epi_log["errors"]["errors"]["relations"]["false_negatives"]

print(f"Count: {len(relations_fn)}\n")
display(relations_fn)

Count: 114



[[28411266, 'sulfonylurea receptor', 'Association', 'type 2 diabetes'],
 [18768591, 'doxorubicin', 'Positive_Correlation', 'ascites'],
 [24743235, 'csf-1', 'Association', 'il-3'],
 [18808529, 'dystrophin', 'Association', 'glycoprotein'],
 [24477591, 'cyba', 'Association', 'nadph oxidases'],
 [18768591, 'sodium', 'Association', 'volume retention'],
 [20510337, 'coenzyme q10', 'Negative_Correlation', 'glutathione'],
 [17192049, 'a to g', 'Negative_Correlation', 'prostate cancer'],
 [18768591, 'nephropathy', 'Positive_Correlation', 'salt'],
 [20510337, 'coenzyme q10', 'Negative_Correlation', 'superoxide dismutase'],
 [18768591,
  'urea',
  'Association',
  'serum- and glucocorticoid-inducible kinase 1'],
 [24341598, 'diltiazem', 'Negative_Correlation', 'calcium'],
 [24477591, '-930a>g', 'Positive_Correlation', 'overweight'],
 [18768591, 'sodium', 'Association', 'nephropathy'],
 [25305591,
  'experimental autoimmune encephalomyelitis',
  'Negative_Correlation',
  'anti-inflammatory cytokin

### Entities

In [30]:
# ---------- False positives ----------
entities_fp = epi_log["errors"]["errors"]["entities"]["false_positives"]

print(f"Count: {len(entities_fp)}\n")
display(entities_fp)

Count: 178



[[21771880, 'bmi', 'DiseaseOrPhenotypicFeature'],
 [10491763, 'mody1', 'GeneOrGeneProduct'],
 [21163864,
  'sporadic hypertrophic cardiomyopathy',
  'DiseaseOrPhenotypicFeature'],
 [17192049, 'estrogens', 'ChemicalEntity'],
 [19108278, 'free fatty acids', 'ChemicalEntity'],
 [18827003,
  'aspartic acid to histidine substitution at amino acid position 401',
  'SequenceVariant'],
 [21163864, 'shcm', 'DiseaseOrPhenotypicFeature'],
 [25305591,
  'pituitary adenylyl cyclase-activating polypeptide',
  'ChemicalEntity'],
 [16574712, 'ecstasy', 'ChemicalEntity'],
 [15970799, '521t>c', 'SequenceVariant'],
 [28428256, 'zo-1', 'GeneOrGeneProduct'],
 [28428256, 'icam1', 'GeneOrGeneProduct'],
 [24914936, 'skeletal dysplasia', 'DiseaseOrPhenotypicFeature'],
 [10491763, 'pro75ala', 'SequenceVariant'],
 [28428256, 'nfkb', 'GeneOrGeneProduct'],
 [10491763, 'insulin', 'GeneOrGeneProduct'],
 [28512644, 'erysipelas', 'DiseaseOrPhenotypicFeature'],
 [20431083, 'is', 'DiseaseOrPhenotypicFeature'],
 [1597079

In [31]:
# ---------- False negatives ----------
entities_fn = epi_log["errors"]["errors"]["entities"]["false_negatives"]

print(f"Count: {len(entities_fn)}\n")
display(entities_fn)

Count: 18



[[16737910, 'cancer', 'DiseaseOrPhenotypicFeature'],
 [15686794, 'antiarrhythmic drug', 'ChemicalEntity'],
 [18827003, 'glucocorticoid', 'ChemicalEntity'],
 [18768591, 'weight gain', 'DiseaseOrPhenotypicFeature'],
 [24743235, 'il-10', 'GeneOrGeneProduct'],
 [15970799, 'tritium', 'ChemicalEntity'],
 [24743235, 'il-3', 'GeneOrGeneProduct'],
 [28411266, 'sulfonylurea receptor', 'GeneOrGeneProduct'],
 [16288197, 'pigmentation', 'DiseaseOrPhenotypicFeature'],
 [18768591, 'salt', 'ChemicalEntity'],
 [24743235, 'csf-1', 'GeneOrGeneProduct'],
 [28411266, 'insulin', 'GeneOrGeneProduct'],
 [20510337, 'lipid', 'ChemicalEntity'],
 [20683499, 'neurodegenerative diseases', 'DiseaseOrPhenotypicFeature'],
 [18808529, 'oxygen', 'ChemicalEntity'],
 [24743235, 'cd11c', 'GeneOrGeneProduct'],
 [18827003, 'cortisol', 'ChemicalEntity'],
 [28348168, 'degenerative disease', 'DiseaseOrPhenotypicFeature']]